In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/fake_jobs_clean.csv')
print(f"Loaded: {len(df)} rows")

Loaded: 17880 rows


In [2]:
df['text'] = (df['title'].fillna('') + ' ' + 
              df['description'].fillna(''))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return ' '.join(text.split())

df['text_clean'] = df['title'].fillna('').str.lower().str.replace(r'[^a-z\s]', ' ', regex=True)
print("Text cleaned")

Text cleaned


In [3]:
print("TF-IDF with 500 features, unigrams only...")
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X_tfidf = tfidf.fit_transform(df['text_clean'])
print(f"Done: {X_tfidf.shape}")

TF-IDF with 500 features, unigrams only...
Done: (17880, 500)


In [4]:
df['text_length'] = df['text_clean'].str.len()
df['has_salary'] = df['salary_range'].notna().astype(int)

from scipy.sparse import hstack
X = hstack([X_tfidf, df[['text_length', 'has_salary', 'has_company_logo', 'has_questions']].values])
y = df['fraudulent'].values

print(f"Final: {X.shape}")

Final: (17880, 504)


In [5]:
import os, pickle
os.makedirs('../src/features', exist_ok=True)
with open('../src/features/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('../src/features/X_y.pkl', 'wb') as f:
    pickle.dump((X, y), f)
print("Saved")

Saved
